###  Spark DataFrame: 
A distributed collection of data organized into named columns, similar to a table in a relational database.
dataframe has data and schema like table in RDBMS, data is organised into columns and rows.

### RDD (Resilient Distributed Dataset):
 A fundamental Spark data structure representing an immutable, distributed collection of objects.

###  Differences:
- DataFrame provides schema, optimizations, and SQL-like operations; RDD is lower-level, unstructured, and supports functional programming.
- DataFrame is optimized by Catalyst and Tungsten; RDD does not benefit from these optimizations.
- DataFrame is easier for data analysis; RDD is used for fine-grained control and custom transformations.

###  Example: Creating DataFrame and RDD

data = [("Alice", 34), ("Bob", 45), ("Cathy", 29)]

### Create DataFrame
df = spark.createDataFrame(data, ["Name", "Age"])

### Create RDD
rdd = spark.sparkContext.parallelize(data)

display(df)
### To view RDD contents:
rdd.collect()


## Transformations and Actions in Apache Spark

### Simple Definition

Spark operations are divided into two categories:

1. **Transformation**
2. **Action**

---

# Easy Way to Remember

Imagine you are cooking.

### Transformation

- Cutting vegetables
- Washing vegetables
- Mixing ingredients

You are **preparing** the food, but you haven't served it yet.

### Action

- Serving the food
- Eating the food

This is when the actual work is completed.

Spark works the same way.

---

# Spark Execution Flow

```text
Read Data
    │
    ▼
Transformation
    │
    ▼
Transformation
    │
    ▼
Transformation
    │
    ▼
Action
    │
    ▼
Spark Executes Everything
```

---

# What is a Transformation?

A **Transformation** creates a **new DataFrame or RDD** from an existing one.

**Important:** Transformations are **lazy**.

That means Spark **does not execute them immediately**.

Instead, Spark remembers the steps and builds an execution plan.

---

## Example

```python
df = spark.read.csv("employees.csv")

filtered_df = df.filter(df.salary > 50000)

selected_df = filtered_df.select("id", "name", "salary")
```

What happened?

- Data was read.
- Records were filtered.
- Columns were selected.

**Did Spark execute the job?**

**No.**

Spark is only preparing the plan.

---

# Visual Representation

```text
employees.csv

        │

        ▼

Read File

        │

        ▼

Filter Salary > 50000

        │

        ▼

Select Columns

        │

        ▼

Nothing Executes Yet!
```

---

# Common Transformations

| Transformation | Purpose |
|----------------|---------|
| select() | Select columns |
| filter() | Filter rows |
| where() | Filter rows |
| withColumn() | Add or modify a column |
| drop() | Remove columns |
| join() | Join DataFrames |
| groupBy() | Group data |
| orderBy() | Sort data |
| distinct() | Remove duplicates |
| dropDuplicates() | Remove duplicate rows |
| union() | Combine DataFrames |
| repartition() | Change number of partitions |
| coalesce() | Reduce partitions |
| map() | Transform RDD elements |
| flatMap() | Flatten collections |

---

# Why Are Transformations Lazy?

Suppose you write:

```python
df = spark.read.csv("sales.csv")

df = df.filter(df.country == "India")

df = df.select("customer_id", "amount")

df = df.withColumn("tax", df.amount * 0.18)
```

Spark does **not** execute each line immediately.

Instead, it creates a plan like this:

```text
Read Sales

↓

Filter India

↓

Select Columns

↓

Calculate Tax

↓

Execution Plan Ready
```

Spark waits until an **Action** is called.

---

# What is an Action?

An **Action** tells Spark:

> "Now execute the entire plan."

This is when Spark:

- Reads the data
- Applies all transformations
- Returns the result
- Writes data to storage if required

---

## Example

```python
df.show()
```

Now Spark executes:

- Read file
- Filter data
- Select columns
- Display results

---

# Visual Representation

```text
Read CSV

↓

Filter

↓

Select

↓

Action (show)

↓

Spark Starts Execution

↓

Result Displayed
```

---

# Common Actions

| Action | Purpose |
|---------|---------|
| show() | Display rows |
| collect() | Bring data to the Driver |
| count() | Count records |
| first() | Return the first row |
| take(n) | Return first n rows |
| head() | Return top rows |
| write() | Save data |
| saveAsTable() | Save as a table |
| foreach() | Execute a function on each row |
| reduce() | Aggregate RDD values |

---

# Real-Time Project Example

Suppose you're processing customer data.

```python
customers = spark.read.parquet("/customer")

customers = customers.filter("status = 'ACTIVE'")

customers = customers.dropDuplicates()

customers = customers.withColumn(
    "country",
    upper(col("country"))
)
```

Has Spark processed the data?

**No.**

Only the execution plan has been built.

Now:

```python
customers.write.parquet("/gold/customer")
```

This is an **Action**.

Spark now executes the entire pipeline.

---

# What Happens Internally?

Suppose you write:

```python
df.filter(...)

.select(...)

.withColumn(...)

.write.parquet(...)
```

Spark creates a **DAG (Directed Acyclic Graph)**.

```text
Read File
     │
     ▼
Filter
     │
     ▼
Select
     │
     ▼
withColumn
     │
     ▼
Write
```

Only when `.write()` is called does Spark execute the DAG.

---

# Real-Time ETL Example

```text
Oracle Database
        │
        ▼
Read Data
        │
        ▼
Filter Active Customers
        │
        ▼
Remove Duplicates
        │
        ▼
Join Country Lookup
        │
        ▼
Calculate Tax
        │
        ▼
Write to Gold Layer
```

Everything before **Write** is a **Transformation**.

**Write** is the **Action**.

---

# Interview Scenario

Suppose you process:

```
500 Million Customer Records
```

Your code:

```python
df = spark.read.parquet("customer")

df = df.filter(df.status == "ACTIVE")

df = df.dropDuplicates()

df = df.select("id", "name", "salary")
```

Nothing has executed yet.

When you run:

```python
df.count()
```

Spark:

- Reads all partitions
- Applies every transformation
- Counts the rows
- Returns the result

---

# Transformation vs Action

| Feature | Transformation | Action |
|----------|----------------|--------|
| Executes immediately? | No | Yes |
| Returns | New DataFrame/RDD | Result or Output |
| Lazy Evaluation | Yes | No |
| Builds DAG | Yes | Uses DAG |
| Triggers Job | No | Yes |

---

# Easy Memory Trick

Imagine washing clothes.

### Transformation

- Put clothes in the washing machine
- Add detergent
- Select wash mode

Nothing is washed yet.

### Action

- Press the **Start** button

Now the machine starts washing.

Spark behaves the same way.

---

# Interview Answer (1–2 Minutes)

> In Spark, operations are divided into **Transformations** and **Actions**. Transformations create a new DataFrame or RDD but do not execute immediately because Spark uses lazy evaluation. Instead, Spark builds an execution plan called a DAG. Common transformations include `filter()`, `select()`, `join()`, and `withColumn()`. An Action, such as `show()`, `count()`, `collect()`, or `write()`, triggers the execution of the entire DAG. In our ETL project, we applied multiple transformations to clean and enrich customer data, and the actual processing started only when we wrote the final dataset to the Gold layer using `write.parquet()`.